# NOTEBOOK 02 — POSTGRESQL DATA PIPELINE
## CSV → PostgreSQL → Aggregation → Join 5 nguồn → Flat Table

### Mục tiêu
- Import 5 CSV vào PostgreSQL.
- Thiết kế schema và index.
- Tối ưu thao tác với bảng hàng triệu dòng bằng `COPY`.
- Aggregate `ByCity` và `ByState` trước khi join để tránh N–N.
- Join 5 nguồn dữ liệu mà vẫn giữ đúng grain của bảng chính.
- Tạo `climate_flat` và `vw_top5_climate_flat`.

> Từ Notebook 03 trở đi, pipeline **không quay lại đọc raw CSV**.

## I. Project setup và kết nối PostgreSQL

In [1]:
from pathlib import Path
import os

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
ARTIFACTS = PROJECT_ROOT / "artifacts"
APP_DIR = PROJECT_ROOT / "app"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
ARTIFACTS.mkdir(parents=True, exist_ok=True)
APP_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = C:\Global Climate Change


In [2]:
from sqlalchemy import create_engine, URL, text

url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password="123456",
    host="localhost",
    port=5432,
    database="climate_change_db1"
)

engine = create_engine(url, pool_pre_ping=True)

with engine.connect() as conn:
    print("✅ Connected")
    print(conn.execute(text("SELECT current_database()")).scalar())

✅ Connected
climate_change_db1


In [3]:
countries = pd.read_sql(
    text("""
        SELECT DISTINCT country
        FROM climate_country
        WHERE country IN (
            'Japan',
            'United States',
            'United Kingdom',
            'France',
            'Australia'
        )
        ORDER BY country;
    """),
    engine
)

display(countries)

NameError: name 'pd' is not defined

## II. Tạo schema raw

Không đặt primary key ngay ở raw layer vì dữ liệu thật có thể chứa duplicate.  
Ta dùng index và kiểm tra business key trước, sau đó mới đưa dữ liệu sạch sang curated layer.

In [ ]:
DDL = '''
DROP TABLE IF EXISTS climate_global CASCADE;
DROP TABLE IF EXISTS climate_country CASCADE;
DROP TABLE IF EXISTS climate_state CASCADE;
DROP TABLE IF EXISTS climate_city CASCADE;
DROP TABLE IF EXISTS climate_major_city CASCADE;

CREATE TABLE climate_global (
    dt DATE,
    land_average_temperature DOUBLE PRECISION,
    land_average_temperature_uncertainty DOUBLE PRECISION,
    land_max_temperature DOUBLE PRECISION,
    land_max_temperature_uncertainty DOUBLE PRECISION,
    land_min_temperature DOUBLE PRECISION,
    land_min_temperature_uncertainty DOUBLE PRECISION,
    land_ocean_average_temperature DOUBLE PRECISION,
    land_ocean_average_temperature_uncertainty DOUBLE PRECISION
);

CREATE TABLE climate_country (
    dt DATE,
    average_temperature DOUBLE PRECISION,
    average_temperature_uncertainty DOUBLE PRECISION,
    country TEXT
);

CREATE TABLE climate_state (
    dt DATE,
    average_temperature DOUBLE PRECISION,
    average_temperature_uncertainty DOUBLE PRECISION,
    state TEXT,
    country TEXT
);

CREATE TABLE climate_city (
    dt DATE,
    average_temperature DOUBLE PRECISION,
    average_temperature_uncertainty DOUBLE PRECISION,
    city TEXT,
    country TEXT,
    latitude TEXT,
    longitude TEXT
);

CREATE TABLE climate_major_city (
    dt DATE,
    average_temperature DOUBLE PRECISION,
    average_temperature_uncertainty DOUBLE PRECISION,
    city TEXT,
    country TEXT,
    latitude TEXT,
    longitude TEXT
);
'''

with engine.begin() as conn:
    conn.execute(text(DDL))

print("Created raw tables.")

Created raw tables.


In [ ]:
import pandas as pd

tables = pd.read_sql(
    text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name;
    """),
    engine
)

display(tables)

,table_name
0,city_country_joined
1,city_temp_cleaned
2,city_temperatures
3,climate_city
4,climate_city_country_month_summary
5,climate_country
6,climate_country_flat
7,climate_flat
8,climate_global
9,climate_major_city


## III. Import 5 CSV bằng PostgreSQL COPY

`COPY FROM STDIN` stream trực tiếp từ file và nhanh hơn `pandas.to_sql()` với bảng 8.6 triệu dòng.  
Notebook không giữ toàn bộ `ByCity` trong RAM.

In [ ]:
FILES = {
    "climate_global": DATA_RAW / "GlobalTemperatures.csv",
    "climate_country": DATA_RAW / "GlobalLandTemperaturesByCountry.csv",
    "climate_state": DATA_RAW / "GlobalLandTemperaturesByState.csv",
    "climate_city": DATA_RAW / "GlobalLandTemperaturesByCity.csv",
    "climate_major_city": DATA_RAW / "GlobalLandTemperaturesByMajorCity.csv",
}

COPY_COLUMNS = {
    "climate_global": [
        "dt", "land_average_temperature", "land_average_temperature_uncertainty",
        "land_max_temperature", "land_max_temperature_uncertainty",
        "land_min_temperature", "land_min_temperature_uncertainty",
        "land_ocean_average_temperature", "land_ocean_average_temperature_uncertainty"
    ],
    "climate_country": [
        "dt", "average_temperature", "average_temperature_uncertainty", "country"
    ],
    "climate_state": [
        "dt", "average_temperature", "average_temperature_uncertainty", "state", "country"
    ],
    "climate_city": [
        "dt", "average_temperature", "average_temperature_uncertainty",
        "city", "country", "latitude", "longitude"
    ],
    "climate_major_city": [
        "dt", "average_temperature", "average_temperature_uncertainty",
        "city", "country", "latitude", "longitude"
    ],
}

def copy_csv_to_postgres(table, csv_path, columns):
    raw_conn = engine.raw_connection()
    try:
        cur = raw_conn.cursor()
        cols = ", ".join(columns)
        sql = f"COPY {table} ({cols}) FROM STDIN WITH (FORMAT CSV, HEADER TRUE)"
        with open(csv_path, "r", encoding="utf-8") as f:
            cur.copy_expert(sql, f)
        raw_conn.commit()
    except Exception:
        raw_conn.rollback()
        raise
    finally:
        raw_conn.close()

for table, path in FILES.items():
    print("Importing", table, "from", path.name)
    copy_csv_to_postgres(table, path, COPY_COLUMNS[table])

print("Import finished.")

Importing climate_global from GlobalTemperatures.csv
Importing climate_country from GlobalLandTemperaturesByCountry.csv
Importing climate_state from GlobalLandTemperaturesByState.csv
Importing climate_city from GlobalLandTemperaturesByCity.csv
Importing climate_major_city from GlobalLandTemperaturesByMajorCity.csv
Import finished.


## IV. Kiểm tra số dòng và missing sau import

In [ ]:
import pandas as pd
from IPython.display import display

tables = list(FILES.keys())
counts = []

with engine.connect() as conn:
    for table in tables:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        counts.append({"table": table, "rows": n})

display(pd.DataFrame(counts).sort_values("rows", ascending=False))

,table,rows
3,climate_city,8599212
2,climate_state,645675
1,climate_country,577462
4,climate_major_city,239177
0,climate_global,3192


## V. Index và business-key audit

Index chính phục vụ các truy vấn join/filter theo `country + dt` hoặc `city + dt`.

In [ ]:
INDEX_SQL = '''
CREATE INDEX IF NOT EXISTS idx_global_dt
ON climate_global(dt);

CREATE INDEX IF NOT EXISTS idx_country_country_dt
ON climate_country(country, dt);

CREATE INDEX IF NOT EXISTS idx_state_country_dt
ON climate_state(country, dt);

CREATE INDEX IF NOT EXISTS idx_city_country_dt
ON climate_city(country, dt);

CREATE INDEX IF NOT EXISTS idx_major_city_city_dt
ON climate_major_city(city, dt);

CREATE INDEX IF NOT EXISTS idx_major_city_country_dt
ON climate_major_city(country, dt);
'''
with engine.begin() as conn:
    conn.execute(text(INDEX_SQL))

dup_query = '''
SELECT COUNT(*) AS duplicate_key_groups
FROM (
    SELECT dt, city, country, COUNT(*) AS n
    FROM climate_major_city
    GROUP BY dt, city, country
    HAVING COUNT(*) > 1
) d;
'''

with engine.connect() as conn:
    print("Duplicate business-key groups:",
          conn.execute(text(dup_query)).scalar())

Duplicate business-key groups: 0


## VI. Aggregate bảng phụ trước khi Join

### Vì sao?
`climate_city` và `climate_state` có nhiều bản ghi cho cùng một `country + dt`.  
Join trực tiếp sẽ biến quan hệ thành N–N và làm phình dữ liệu.

Ta aggregate về đúng grain `Country × Month`.

In [ ]:
AGG_SQL = '''
DROP TABLE IF EXISTS climate_city_country_month_summary;
CREATE TABLE climate_city_country_month_summary AS
SELECT
    dt,
    country,
    AVG(average_temperature) AS city_country_avg_temperature,
    AVG(average_temperature_uncertainty) AS city_country_avg_uncertainty,
    COUNT(*) AS city_records
FROM climate_city
GROUP BY dt, country;

DROP TABLE IF EXISTS climate_state_country_month_summary;
CREATE TABLE climate_state_country_month_summary AS
SELECT
    dt,
    country,
    AVG(average_temperature) AS state_country_avg_temperature,
    AVG(average_temperature_uncertainty) AS state_country_avg_uncertainty,
    COUNT(*) AS state_records
FROM climate_state
GROUP BY dt, country;

CREATE INDEX IF NOT EXISTS idx_city_summary_country_dt
ON climate_city_country_month_summary(country, dt);

CREATE INDEX IF NOT EXISTS idx_state_summary_country_dt
ON climate_state_country_month_summary(country, dt);
'''
with engine.begin() as conn:
    conn.execute(text(AGG_SQL))

summary_counts = pd.read_sql(text('''
SELECT 'city_summary' AS table_name, COUNT(*) AS rows
FROM climate_city_country_month_summary
UNION ALL
SELECT 'state_summary', COUNT(*)
FROM climate_state_country_month_summary;
'''), engine)
display(summary_counts)

,table_name,rows
0,city_summary,393585
1,state_summary,19051


## VII. Join 5 nguồn và tạo Flat Table

Nguồn:
1. `climate_major_city` — main
2. `climate_country`
3. `climate_global`
4. `climate_city_country_month_summary`
5. `climate_state_country_month_summary`

Latitude/Longitude được chuyển sang số tại SQL layer.

In [ ]:

FLAT_SQL = '''
DROP VIEW IF EXISTS vw_top5_climate_flat;
DROP TABLE IF EXISTS climate_flat;

CREATE TABLE climate_flat AS
SELECT
    mc.dt,
    mc.average_temperature,
    mc.average_temperature_uncertainty,
    mc.city,
    mc.country,
    mc.latitude,
    mc.longitude,

    CASE
        WHEN mc.latitude IS NULL THEN NULL
        ELSE CAST(REGEXP_REPLACE(mc.latitude, '[^0-9.]', '', 'g') AS DOUBLE PRECISION)
             * CASE WHEN mc.latitude LIKE '%S' THEN -1 ELSE 1 END
    END AS latitude_num,

    CASE
        WHEN mc.longitude IS NULL THEN NULL
        ELSE CAST(REGEXP_REPLACE(mc.longitude, '[^0-9.]', '', 'g') AS DOUBLE PRECISION)
             * CASE WHEN mc.longitude LIKE '%W' THEN -1 ELSE 1 END
    END AS longitude_num,

    c.average_temperature AS country_temperature,
    c.average_temperature_uncertainty AS country_uncertainty,

    g.land_average_temperature AS global_temperature,
    g.land_average_temperature_uncertainty AS global_uncertainty,

    cs.city_country_avg_temperature,
    cs.city_country_avg_uncertainty,
    cs.city_records,

    ss.state_country_avg_temperature,
    ss.state_country_avg_uncertainty,
    ss.state_records

FROM climate_major_city mc

LEFT JOIN climate_country c
    ON mc.dt = c.dt
   AND mc.country = c.country

LEFT JOIN climate_global g
    ON mc.dt = g.dt

LEFT JOIN climate_city_country_month_summary cs
    ON mc.dt = cs.dt
   AND mc.country = cs.country

LEFT JOIN climate_state_country_month_summary ss
    ON mc.dt = ss.dt
   AND mc.country = ss.country;
'''
with engine.begin() as conn:
    conn.execute(text(FLAT_SQL))
    conn.execute(text(
        "CREATE INDEX IF NOT EXISTS idx_climate_flat_city_dt "
        "ON climate_flat(city, dt)"
    ))

print("Created climate_flat.")

Created climate_flat.


## VIII. Validate Join

Điều kiện bắt buộc:

> `COUNT(climate_flat) == COUNT(climate_major_city)`

Nếu khác, join đã nhân hoặc làm mất grain.

In [ ]:
validation = pd.read_sql(text('''
SELECT
    (SELECT COUNT(*) FROM climate_major_city) AS main_rows,
    (SELECT COUNT(*) FROM climate_flat) AS flat_rows,
    (SELECT COUNT(*) FROM climate_flat WHERE country_temperature IS NULL)
        AS missing_country_context,
    (SELECT COUNT(*) FROM climate_flat WHERE global_temperature IS NULL)
        AS missing_global_context,
    (SELECT COUNT(*) FROM climate_flat WHERE city_country_avg_temperature IS NULL)
        AS missing_city_context,
    (SELECT COUNT(*) FROM climate_flat WHERE state_country_avg_temperature IS NULL)
        AS missing_state_context;
'''), engine)

display(validation)

assert validation.loc[0, "main_rows"] == validation.loc[0, "flat_rows"], \
    "Join làm thay đổi số dòng bảng chính."

,main_rows,flat_rows,missing_country_context,missing_global_context,missing_city_context,missing_state_context
0,239177,239177,16621,1118,5980,130442


## IX. View phạm vi 5 thành phố và >100 năm

Filter dữ liệu từ 1850 để thống nhất horizon lịch sử.

In [ ]:
TARGET_CITIES = ["Tokyo", "New York", "London", "Paris", "Sydney"]

with engine.begin() as conn:
    conn.execute(text("DROP VIEW IF EXISTS vw_top5_climate_flat"))
    conn.execute(text('''
        CREATE VIEW vw_top5_climate_flat AS
        SELECT *
        FROM climate_flat
        WHERE city IN ('Tokyo', 'New York', 'London', 'Paris', 'Sydney')
          AND dt >= DATE '1850-01-01';
    '''))

preview = pd.read_sql(
    text("SELECT * FROM vw_top5_climate_flat ORDER BY city, dt LIMIT 20"),
    engine
)
display(preview)

,dt,average_temperature,average_temperature_uncertainty,city,country,latitude,longitude,latitude_num,longitude_num,country_temperature,country_uncertainty,global_temperature,global_uncertainty,city_country_avg_temperature,city_country_avg_uncertainty,city_records,state_country_avg_temperature,state_country_avg_uncertainty,state_records
0,1850-01-01,-0.181,1.750,London,United Kingdom,52.24N,0.00W,52.24,-0.0,1.125,1.565,0.749,1.105,0.506176,1.605221,68,None,None,None
1,1850-02-01,6.336,1.481,London,United Kingdom,52.24N,0.00W,52.24,-0.0,6.230,1.509,3.071,1.275,6.321824,1.506515,68,None,None,None
2,1850-03-01,4.136,2.920,London,United Kingdom,52.24N,0.00W,52.24,-0.0,4.645,2.996,4.954,0.955,4.355632,2.985397,68,None,None,None
3,1850-04-01,8.758,2.167,London,United Kingdom,52.24N,0.00W,52.24,-0.0,7.622,2.076,7.217,0.665,8.216779,2.113853,68,None,None,None
4,1850-05-01,10.446,0.889,London,United Kingdom,52.24N,0.00W,52.24,-0.0,8.915,0.953,10.004,0.617,9.770500,0.875779,68,None,None,None
5,1850-06-01,15.784,0.961,London,United Kingdom,52.24N,0.00W,52.24,-0.0,13.451,1.047,13.150,0.614,14.857985,0.992956,68,None,None,None
6,1850-07-01,16.486,0.991,London,United Kingdom,52.24N,0.00W,52.24,-0.0,14.143,1.047,14.492,0.614,15.562456,1.008294,68,None,None,None
7,1850-08-01,15.465,1.124,London,United Kingdom,52.24N,0.00W,52.24,-0.0,13.076,1.068,14.039,0.802,14.484412,1.035426,68,None,None,None
8,1850-09-01,12.603,1.152,London,United Kingdom,52.24N,0.00W,52.24,-0.0,10.933,1.084,11.505,0.675,11.983044,1.035676,68,None,None,None
9,1850-10-01,7.702,0.967,London,United Kingdom,52.24N,0.00W,52.24,-0.0,7.167,1.053,8.091,0.863,7.647971,0.944118,68,None,None,None


## X. Performance check
Dùng `EXPLAIN ANALYZE` để kiểm tra query theo city/date có tận dụng index hay không.

In [ ]:
with engine.connect() as conn:
    plan = conn.execute(text('''
        EXPLAIN ANALYZE
        SELECT *
        FROM climate_flat
        WHERE city = 'Tokyo'
          AND dt BETWEEN DATE '1900-01-01' AND DATE '2013-12-31'
        ORDER BY dt;
    ''')).fetchall()

print("\n".join(row[0] for row in plan))

Sort  (cost=28.06..28.08 rows=6 width=244) (actual time=0.614..0.656 rows=1365.00 loops=1)
  Sort Key: dt
  Sort Method: quicksort  Memory: 251kB
  Buffers: shared hit=58 read=7
  ->  Bitmap Heap Scan on climate_flat  (cost=4.50..27.99 rows=6 width=244) (actual time=0.239..0.396 rows=1365.00 loops=1)
        Recheck Cond: ((city = 'Tokyo'::text) AND (dt >= '1900-01-01'::date) AND (dt <= '2013-12-31'::date))
        Heap Blocks: exact=57
        Buffers: shared hit=58 read=7
        ->  Bitmap Index Scan on idx_climate_flat_city_dt  (cost=0.00..4.50 rows=6 width=0) (actual time=0.218..0.218 rows=1365.00 loops=1)
              Index Cond: ((city = 'Tokyo'::text) AND (dt >= '1900-01-01'::date) AND (dt <= '2013-12-31'::date))
              Index Searches: 1
              Buffers: shared hit=1 read=7
Planning Time: 0.115 ms
Execution Time: 0.749 ms


## XI. Kết luận Notebook 02

Đầu ra:
- raw tables cho 5 CSV;
- 2 bảng summary;
- `climate_flat` join đủ 5 nguồn;
- `vw_top5_climate_flat` làm nguồn cho Notebook 03;
- index cho city/date và country/date;
- kiểm tra join không làm thay đổi số dòng main table.

**Notebook 03 chỉ đọc PostgreSQL.**